# 1. Import Libraries

In [2]:
!pip install google-play-scraper nltk

from google_play_scraper import reviews, Sort
from pprint import pprint
import pathlib
import csv
import os
from nltk.corpus import stopwords


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


# 2. Create Scraping Function

In [ ]:
app_id : str = "com.valvesoftware.android.steam.community"
scrape_review_count : int = 10000
raw_data_path = "../data/raw/"
reviews_csv_file_template : str = os.path.join(raw_data_path, "reviews-{}-stars.csv")

def scrape_reviews(
        count : int = 100,
        score : int = None,
        sort  : Sort = Sort.MOST_RELEVANT, 
        lang  : str = "en",
    ):

    result = reviews(
        app_id = app_id,
        lang   = lang,
        sort   = sort,
        count  = count,
        filter_score_with  = score,
    )

    return result

def output_csv(data: list, file):
    if (len(data) > 0):
        writer = csv.writer(file)

        header_keys = data[0].keys()
        writer.writerow(header_keys)
        
        for review in data:
            writer.writerow(review.values())

# 3. Scrape Data

In [4]:
# test
data, _ = scrape_reviews(count=1)

pprint(data)

[{'appVersion': '3.10.9',
  'at': datetime.datetime(2026, 4, 12, 9, 16, 17),
  'content': 'The Steam app was working fine for years, it logged me out '
             'recently and whwn I went to log back in it doesnt work half the '
             'time. When I do get in and try to view my wishlist or games in '
             'the store it gives me an error mesaage that says "There was an '
             'error communicating with the Steam servers. Please try again '
             'later." It\'s been like this for a couple weeks now. If this '
             "continues through the weekend I'm convering everything to Epic "
             'and Microsoft for my games. This is becoming too much of a '
             'hassle.',
  'repliedAt': None,
  'replyContent': None,
  'reviewCreatedVersion': '3.10.9',
  'reviewId': '52b149e5-27b3-42c5-ab97-3b94cfaf672c',
  'score': 1,
  'thumbsUpCount': 1,
  'userImage': 'https://play-lh.googleusercontent.com/a-/ALV-UjXwgw-vFOQvF6v6GDQeWhC9Gm5_GqDT93Q5zMhmjejgOp

# 4. Save to CSV

In [5]:
# scrape and store to csv
data, _ = scrape_reviews(count=scrape_review_count)

# make content lowercase

for review in data:
    review.pop("userName", None)
    review.pop("userImage", None)

    if (review["content"] == None):
        continue
    
    content_lowercase = review["content"].lower()

    review["content"] = content_lowercase

# mkdir the parent if not exists
dir_parent = pathlib.Path(reviews_csv_file_template).parent
dir_parent.mkdir(parents=True, exist_ok=True)

reviews_csv_file_dir_star = reviews_csv_file_template.format("mixed")
reviews_csv_file = open(reviews_csv_file_dir_star, "w")

# write list of dicts as a csv into file
output_csv(data=data, file=reviews_csv_file)

# close file
reviews_csv_file.close()

# 5. Analyse Reviews

In [6]:
review_stars_count = {
    1: 0,
    2: 0,
    3: 0,
    4: 0,
    5: 0
}

total_reviews = 0
sum_score = 0

for review in data:
    score = review["score"]
    if (score not in review_stars_count):
        review_stars_count[score] = 1
    else:
        review_stars_count[score] += 1
    
    total_reviews += 1
    sum_score += score

print(f"Sum score: {sum_score}")
print(f"Avg score: {sum_score / total_reviews}")
print(f"Score counts: {review_stars_count}")

Sum score: 17837
Avg score: 1.7837
Score counts: {1: 6460, 2: 1306, 3: 908, 4: 589, 5: 737}
